<a href="https://colab.research.google.com/github/munozsan03/Astro_Team4/blob/feature%2Fdata-pipeline/Astro_Team4_Data_Pipeline_Torchlight_Hackathon_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to the 2026 Torchlight Hackathon!

Your challenge for this hackathon is to help astronauts better understand the effects their space travel has had on their overall wellbeing. You'll use the individual astronaut data from the [2021 Inspiration 4 mission](https://inspiration4.com/).

In this notebook, you can access, read in, and download data files from the Inspiration 4 astronauts and their spacecraft, using the [Developer API](https://www.nasa.gov/reference/osdr-developer-api/) from the [NASA Open Science Data Repository](https://science.nasa.gov/biological-physical/data/osdr/).

In [5]:
import os, re
import pandas as pd

# ── Timepoint normalisation ───────────────────────────────────────────────────
# Maps every messy variant found across the 19 datasets → (canonical, phase)
_TP_MAP = {
    "l-92":"L-92","l_92":"L-92","l92":"L-92",
    "l-44":"L-44","l_44":"L-44","l44":"L-44",
    "l-3":"L-3","l_3":"L-3","l3":"L-3",
    "fd2":"FD2","fd_2":"FD2","fd 2":"FD2","day2":"FD2",
    "fd3":"FD3","fd_3":"FD3","fd 3":"FD3","day3":"FD3",
    "r+1":"R+1","r_1":"R+1","r1":"R+1",
    "r+45":"R+45","r_45":"R+45","r45":"R+45",
    "r+82":"R+82","r_82":"R+82","r82":"R+82",
}
_PHASE_MAP = {
    "L-92":"Preflight","L-44":"Preflight","L-3":"Preflight",
    "FD2":"Inflight","FD3":"Inflight",
    "R+1":"Postflight","R+45":"Postflight","R+82":"Postflight",
}
CANONICAL_TIMEPOINTS = ["L-92","L-44","L-3","FD2","FD3","R+1","R+45","R+82"]

def standardize_timepoint(val):
    """Return canonical timepoint string, or 'UNKNOWN'."""
    s = str(val).lower().strip()
    if s in _TP_MAP:
        return _TP_MAP[s]
    for k, v in _TP_MAP.items():
        if k in s:
            return v
    return "UNKNOWN"

def get_phase(tp):
    return _PHASE_MAP.get(tp, "UNKNOWN")

# ── Crew ID normalisation ─────────────────────────────────────────────────────
# Handles: C001, CM02, crewmember3, crewID4, SUBJECT_ID_01, SUB02, S3, M4 …
_CREW_RE = re.compile(
    r"(?:C(?:M|rew(?:member|ID)?|rw)?[-_]?0*([1-4])"
    r"|(?:SUBJECT(?:_ID)?|SUB|MEMBER)[-_]?0*([1-4])"
    r"|(?:S|M)[-_]?0*([1-4])(?!\d))",
    re.IGNORECASE,
)

def standardize_crew_id(val):
    """Return canonical crew ID (C001–C004), or 'UNKNOWN'."""
    m = _CREW_RE.search(str(val))
    if m:
        digit = next(g for g in m.groups() if g is not None)
        return f"C{int(digit):03d}"
    return "UNKNOWN"

# ── Column-header parser (wide metagenomics tables) ───────────────────────────
_TP_COL_RE = re.compile(r"(L[-_]92|L[-_]44|L[-_]3|FD[-_]?[23]|R[+_]\d+)", re.IGNORECASE)

def parse_col_meta(col):
    """Extract (crew_id, timepoint, phase) from a column name string."""
    crew = standardize_crew_id(col)
    m = _TP_COL_RE.search(str(col))
    tp   = standardize_timepoint(m.group(1)) if m else "UNKNOWN"
    return crew, tp, get_phase(tp)

def add_col_metadata(df):
    """
    For wide tables (rows=features, cols=samples): attach a .attrs['meta']
    DataFrame with crew_id / timepoint / phase for every sample column.
    """
    rows = []
    for col in df.columns:
        crew, tp, phase = parse_col_meta(str(col))
        rows.append({"sample_col": col, "crew_id": crew,
                     "timepoint": tp, "phase": phase})
    df.attrs["meta"] = pd.DataFrame(rows).set_index("sample_col")
    return df

def tag_long(df, crew_col=None, timepoint_col=None):
    """
    For long tables (rows=samples): normalise crew_id / timepoint / phase
    columns in-place.
    crew_col      – existing column whose values contain the raw crew ID
    timepoint_col – existing column whose values contain the raw timepoint
    """
    df = df.copy()
    if crew_col and crew_col in df.columns:
        df["crew_id"] = df[crew_col].apply(standardize_crew_id)
    elif "crew_id" not in df.columns:
        # Try to extract from the DataFrame index
        df["crew_id"] = df.index.to_series().apply(standardize_crew_id)

    if timepoint_col and timepoint_col in df.columns:
        df["timepoint"] = df[timepoint_col].apply(standardize_timepoint)
    elif "timepoint" not in df.columns:
        df["timepoint"] = df.index.to_series().apply(standardize_timepoint)

    df["phase"] = df["timepoint"].apply(get_phase)
    return df

print("✅ Standardisation helpers loaded.")
print(f"   Canonical timepoints: {CANONICAL_TIMEPOINTS}")


✅ Standardisation helpers loaded.
   Canonical timepoints: ['L-92', 'L-44', 'L-3', 'FD2', 'FD3', 'R+1', 'R+45', 'R+82']


In [6]:
import os
os.makedirs("data/processed", exist_ok=True)

exports = {
    # ── Direct analyte panels ──────────────────────────────────────────────
    "cmp_metabolic_panel":          metabolic_blood,
    "cbc_complete_blood_count":     cbc,
    "urine_inflammation_panel":     urine,
    "immune_cytokines_eve":         immune_eve,
    "immune_cytokines_alamar":      immune_alamar,
    "cardiac_cytokines_eve":        cardio_eve,
    # ── Omics panels ───────────────────────────────────────────────────────
    "plasma_metabolomics":          metabolomics,
    "evp_proteomics":               evp,
    "plasma_proteomics":            protein,
    "whole_blood_rnaseq":           rna_blood,
    "whole_blood_m6A":              m6A,
    "pbmc_snrnaseq":                snrnaseq,
    "pbmc_snATACseq":               snatacseq,
    "pbmc_vdj":                     vdj,
    "skin_spatial_transcriptomics": spatial,
    # ── Metagenomics ───────────────────────────────────────────────────────
    "crew_swab_taxonomy":           tax_metagenomics_crew,
    "stool_taxonomy":               tax_metagenomics_stool,
    "skin_taxonomy":                tax_metagenomics_skin,
    "dragon_capsule_taxonomy":      tax_metagenomics_dragon,
}

for name, df in exports.items():
    path = f"data/processed/{name}.csv"
    df.to_csv(path)
    # For wide tables, also save the column metadata sidecar
    if hasattr(df, "attrs") and "meta" in df.attrs:
        df.attrs["meta"].to_csv(f"data/processed/{name}__meta.csv")
    print(f"✅ {name} → {path}  ({len(df)} rows × {len(df.columns)} cols)")

print("\nAll 19 datasets written to data/processed/")


NameError: name 'metabolic_blood' is not defined

## 1. Oral, Nasal, and Skin Metagenomic Microbial Swabs

Dataset ID: [OSD-572](https://osdr.nasa.gov/bio/repo/data/studies/OSD-572)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included oral, nasal, and skin microbial swabs collected from ten locations (oral, nasal cavity, post-auricular, axillary vault, volar forearm, occiput, umbilicus, gluteal crease, glabella, and toe web space). Swabs were collected at three pre-flight timespoints (L-92, L-44, L-3), two in-flight timepoints (flight day 2 (FD2), and flight day 3 (FD3)), and three post-flight timepoints (R+1, R+45, R+82).

There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [ ]:
kegg_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Combined-gene-level-KO-function-coverages_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_crew = add_col_metadata(kegg_metagenomics_crew)
kegg_metagenomics_crew.head()

### Taxonomy Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different microbial taxon that has been quantified within that sample.

In [ ]:
tax_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_crew = add_col_metadata(tax_metagenomics_crew)
tax_metagenomics_crew.head()

### Pathway Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [ ]:
pathway_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_crew = add_col_metadata(pathway_metagenomics_crew)
pathway_metagenomics_crew.head()

### Gene Family Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different gene family whose activity was quantified within that sample.

In [ ]:
genefam_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_crew = add_col_metadata(genefam_metagenomics_crew)
genefam_metagenomics_crew.head()

# 2. Blood Serum Metabolic Panel

Dataset ID: [OSD-575](https://osdr.nasa.gov/bio/repo/data/studies/OSD-575)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, with serum extracted from blood using a serum separator tube (SST). Samples were collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Serum samples were submitted to Quest Diagnostics for comprehensive metabolic panel testing.

Each column is a serum sample taken from a crew member.

Each row is a metabolic measurement from Quest.

In [ ]:
metabolic_blood = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv',
    index_col=0).transpose()
metabolic_blood = tag_long(metabolic_blood)
metabolic_blood.head()

# 3. Immune/Cardiac Cytokine Arrays

Dataset ID: [OSD-575](https://osdr.nasa.gov/bio/repo/data/studies/OSD-575)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, with serum extracted from blood using a serum separator tube (SST). Samples were collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Serum samples were submitted for immune and cardiovascular cytokine biomarker profiling panels at two different companies (Eve Technologies and Alamar).

Below we read in 3 different files:

* Immune cytokine biomarkers quantified by Eve Technologies
* Immune cytokine biomakers quantified by Alamar
* Cardiac cytokine biomarkers quantified by Eve Technologies


In each file, the columns are serum samples from crew members, and the rows are cytokine measurements from Eve or Alamar.

In [ ]:
immune_eve = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFORMED.csv',
    index_col=0).transpose()
immune_eve = tag_long(immune_eve)
immune_eve.head()

# New Section

In [ ]:
immune_alamar = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum.immune.AlamarPanel_TRANSFORMED.csv',
    index_col=0).transpose()
immune_alamar = tag_long(immune_alamar)
immune_alamar.head()

In [ ]:
cardio_eve = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum_cardiovascular_EvePanel_TRANSFORMED.csv',
    index_col=0).transpose()
cardio_eve = tag_long(cardio_eve)
cardio_eve.head()

# 4. Dragon Capsule Metagenomic Microbial Swabs

Dataset ID: [OSD-573](https://osdr.nasa.gov/bio/repo/data/studies/OSD-573)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included microbial swabs collected from nine surfaces in the Dragon capsule (execute button, G-meter button, control touch screen - left, control touch screen - right, side hatch mobility aid, lid of waste locker, seat 2, commode panel, viewing dome) and one open air control. Swabs were collected twice during flight (flight day 2 (FD2), and flight day 3 (FD3)) and twice pre-flight in the crew training capsule in Hawthorne, CA (L-92, L-44).

There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [ ]:
kegg_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_dragon = add_col_metadata(kegg_metagenomics_dragon)
kegg_metagenomics_dragon.head()

### Taxonomy Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [ ]:
tax_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_dragon = add_col_metadata(tax_metagenomics_dragon)
tax_metagenomics_dragon.head()

### Pathway Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [ ]:
pathway_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_dragon = add_col_metadata(pathway_metagenomics_dragon)
pathway_metagenomics_dragon.head()

### Gene Family Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different gene family whose activity was quantified within that sample.

In [ ]:
genefam_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_dragon = add_col_metadata(genefam_metagenomics_dragon)
genefam_metagenomics_dragon.head()

# 5. Metabolomics

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used for metabolomics; metabolite abundances were calculated pre vs post-flight to generate processed data files.

Notes on processed data:

* Mission	Inspiration4
* Data	Plasma Metabolomics
* Database Identifier	I4-FP1
* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are proteins.

In [ ]:
metabolomics = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_metabolomics_Plasma_Metabolomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
metabolomics.head()

# 6. EVPs (Extracellular Vesicles and Particles) for Mass Spectrometry

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used to isolate extracellular vesicles and particles (EVPs) for mass spectrometry.

Notes on processed data:

* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are proteins quantified from the EVPs through mass spectrometry.



In [ ]:
evp = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_proteomics_EVP_Proteomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
evp.head()

# 7. Proteomics

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used for proteomics; differential protein abundances were calculated pre vs post-flight to generate processed data files.

Notes on processed data:

* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are protein abundances.


In [ ]:
protein = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_proteomics_Plasma_Proteomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
protein.head()

# 8. Urine Inflammation Panel (Multiplex, NULISAseq)

Dataset ID: [OSD-656](https://osdr.nasa.gov/bio/repo/data/studies/OSD-656)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included urine from pre-flight and post-flight/recovery timepoints (L-92, L-44, L-3, R+1, R+45, R+82). 203 inflammatory, cytokine, and chemokine proteins were quantified using NULISAseq. This study derives results from the Multiplex assay.

The columns are quantifications of inflammatory proteins; the rows are samples from crew members at specific time points.

In [ ]:
urine = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-656/download?source=datamanager'
    '&file=LSDS-64_Multiplex_urine.immune.AlamarPanel_TRANSFORMED.csv',
    index_col=0)
urine = tag_long(urine)
urine.head()

# 9. Stool Metagenome Profiling

Dataset ID: [OSD-630](https://osdr.nasa.gov/bio/repo/data/studies/OSD-630)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included stool collected in OMNIgene•GUT tubes (DNA Genotek, OMR-200). Samples were collected pre-flight (L-92, L-44) and post-flight (R+45, R+82).

There are several different ways to quantify how the microbial populations are changing in each sample at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each sample:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [ ]:
kegg_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_stool = add_col_metadata(kegg_metagenomics_stool)
kegg_metagenomics_stool.head()

### Taxonomy Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [ ]:
tax_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_stool = add_col_metadata(tax_metagenomics_stool)
tax_metagenomics_stool.head()

### Pathway Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [ ]:
pathway_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_stool = add_col_metadata(pathway_metagenomics_stool)
pathway_metagenomics_stool.head()

### Gene Family Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different gene family whose activity was quantified within that sample.

In [ ]:
genefam_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_stool = add_col_metadata(genefam_metagenomics_stool)
genefam_metagenomics_stool.head()

# 10. Whole Blood Profiling

Dataset ID: [OSD-569](https://osdr.nasa.gov/bio/repo/data/studies/OSD-569)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82, R+194). Total RNA was purified from each tube and mRNA was sequenced. Data was used to generate differential gene expression profiles and annotate sites of m6A modification. Also, whole blood was submitted to Quest Diagnostics for a complete blood count at all ground timepoints (L-92, L-44, L-3, R+1, R+45, R+82, R+194).

### Total RNA Sequencing

Notes on processed data:
*   Comparison: (R+82) vs (L-92, L-44, L-3)
*   Data Note 1: FeatureCounts was used to calculate DEGs with DESeq2. Salmon was used with pipeline-transcriptome-de.
*   Data Note 2: Positive logFC = higher abundance at (R+82); Negative logFC = lower abundance at (R+82).


The rows are genes; the columns are samples from crew members at specific time poionts.




In [ ]:
rna_blood = pd.read_excel(
    "https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager"
    "&file=GLDS-561_long-readRNAseq_Direct_RNA_seq_Gene_Expression_Processed.xlsx",
    skiprows=[0,1,2,3,4,5,6,9], header=[0,1], index_col=0)
rna_blood.columns = [
    f"{str(a)}_{str(b)}".strip("_") for a, b in rna_blood.columns
]
rna_blood = add_col_metadata(rna_blood)
rna_blood.head()

### m6A Modification

m6A is a chemical modification on mRNA that affects how transcripts are regulated, degraded, and translated. Measuring it in whole blood across timepoints reveals how spaceflight stress alters post-transcriptional gene regulation.

Notes on processed data:
* Comparison: (R+1) vs (L-92, L-44, L-3)
* Data Note 1: Calculated with m6Anet (v1.1.1) and methyl-kit (v1.22.0)
* Data Note 2: Position refers to the location on the transcript, relative from the 5' transcriptional start site.
* Data Note 3: Crew member/timepoint columns refers to the probability that within the indicated sample the position within that particular transcript contains an m6a modification. NA denotes positions within samples that lacked coverage to make a prediction (i.e. less than twenty reads per the definition of m6Anet).
* Data Note 4: Higher or lower abundance postflight is indicated by positive or negative methylKit methylation differences, respectively, for the given comparison (indicated within column W).


The rows are gene and transcript names; the columns are samples from crew members at specific time points.


In [ ]:
m6A = pd.read_excel(
    "https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager"
    "&file=GLDS-561_directm6Aseq_Direct_RNA_seq_m6A_Processed_Data.xlsx",
    skiprows=[0,1,2,3,4,5,6,7], header=[0,1], index_col=0)
m6A.columns = [
    f"{str(a)}_{str(b)}".strip("_") for a, b in m6A.columns
]
m6A.index.name = "transcript_ENSEMBL"
m6A = add_col_metadata(m6A)
m6A.head()

### Complete Blood Count (CBC)

Each row is a measurement from the CBC assay. Each column are the values and units, as well as the crew member (SUBJECT_ID) and time point (TEST_DATE).


In [ ]:
cbc = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager'
    '&file=LSDS-7_Complete_Blood_Count_CBC.upload_SUBMITTED.csv',
    index_col=0)
cbc = tag_long(cbc, crew_col="SUBJECT_ID", timepoint_col="TEST_DATE")
cbc.head()

# 11. PBMC (Peripheral Blood Mononuclear Cells) Profiling

Dataset ID: [OSD-570](https://osdr.nasa.gov/bio/repo/data/studies/OSD-570)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included PBMCs collected via venipuncture, collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Single-nuclei RNA-seq and single-nuclei ATAC-seq were co-assayed (L-92, L-44, L-3, R+1, R+45, R+82 timepoints). T-cell and B-cell V(D)J repertoires were profiled (L-3, R+1, R+45, R+82 timepoints). Differential gene expression profiles and clonotypes detected per timepoint were calculated from sequencing results.


PBMCs are  the immune cells circulating in blood, and profiling them reveals how the immune system is adapting or dysregulating in real time.

### Single-nuclei RNA-seq

Notes on processed data:

* Comparison:	(R+45) vs (R+1)
* Data Note 1:	Output of Seurat (v4.2.0) FindMarkers with a log fold change (avg_log2FC) cutoff point of 0.25.
* Data Note 2:	FindMarkers was calculated separately for each cell type. All cell types have been appended to this sheet.
* Data Note 3:	Positive avg_log2FC = higher gene expression at (R+45); Negative avg_log2FC = lower gene expression at (R+45).

The rows are genes; the columns are statistics from the FindMarkers analysis.

In [ ]:
snrnaseq = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_snRNA-Seq_PBMC_Gene_Expression_snRNA-seq_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
snrnaseq.head()

### Single-nuclei ATAC-seq

Notes on processed data:
* Comparison:	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Output of Seurat (v4.2.0) FindMarkers with a log fold change (avg_log2FC) cutoff point of 0.25.
* Data Note 2:	FindMarkers was calculated separately for each cell type. All cell types have been appended to this sheet.
* Data Note 3:	Positive avg_log2FC = higher peak at (R+1); Negative avg_log2FC = lower peak at (R+1).

Each row is a chromatin accessibility peak defined by its chromosome and start/end coordinates; the columns are statistics from the FindMarkers analysis.

In [ ]:
snatacseq = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_snATAC-Seq_PBMC_Chromatin_Accessibility_snATAC-seq_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
snatacseq.head()

### T-cell and B-cell VDJ Profiles

V(D)J profiles capture the unique DNA rearrangements that give each T and B cell its specific receptor. Sequencing these from PBMCs tells you which immune cell clones are expanding or contracting over the course of spaceflight, revealing whether the immune system is mounting specific responses, losing diversity, or shifting in ways that could indicate immune dysfunction.


Notes from processed data:
* Data note: Calculated with VGenes package (https://github.com/WilsonImmunologyLab/Vgenes)

Each row is a single TCR clone identified from a crew member at a specific timepoint, with its alpha or beta chain sequence and any somatic mutations relative to the germline; the columns contain additional information and quantification as well as the crew member (crewID) and timepoint.


In [ ]:
vdj = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_scRNA-Seq_VDJ_Results.xlsx',
    skiprows=[0,1,2], index_col=0)
vdj = tag_long(vdj, crew_col="crewID", timepoint_col="timepoint")
vdj.head()

# 12. Deltoid Skin and Microbiome Data

Dataset ID: [OSD-574](https://osdr.nasa.gov/bio/repo/data/studies/OSD-574)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included deltoid skin biopsies, collected once pre-flight (L-44) and within 1 day postflight (R+1). Samples were used for **spatially resolved transcriptomics** using the NanoString GeoMx Whole Transcriptome Atlas Panel. Skin was profiled in four regions: outer epidermis (OE), inner epidermis (IE), outer dermis (OD), and vasculature (VA) regions. Four crew members were profiled pre-fight (C001, C002, C003, C004) and three crew members were profiled post-flight (C002, C003, C004).

Additionally, skin swabs of the deltoid region were collected before each biopsy. DNA was extracted from the swab to generate **metagenomic profiles**.
There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).


### Spatial transcriptomics differential gene expression analysis

Notes on processed data:
* Comparison:	(R+1) vs (L-44)
* Data Note 1:	Positive logFC = higher expression post-flight; Negative logFC = lower expression post-flight.
* Data Note 2:	GeoMx Whole Transcriptome Atlas Panel (WTA, 18,677 genes)
* Data Note 3:	OE = Outer Epidermis, IE = Inner Epidermis, OD = Outer Dermis, VA = Vasculature



In [ ]:
spatial = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-566_SpatialTranscriptomics_Skin_Biopsy_Spatially_Resolved_Transcriptomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
spatial.head()

### KEGG Orthology (KO) Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [ ]:
kegg_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_skin = add_col_metadata(kegg_metagenomics_skin)
kegg_metagenomics_skin.head()

### Taxonomy Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [ ]:
tax_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_skin = add_col_metadata(tax_metagenomics_skin)
tax_metagenomics_skin.head()

### Pathway Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [ ]:
pathway_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_skin = add_col_metadata(pathway_metagenomics_skin)
pathway_metagenomics_skin.head()

### Gene Family Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different gene family whose activity was quantified within that sample.

In [ ]:
genefam_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_skin = add_col_metadata(genefam_metagenomics_skin)
genefam_metagenomics_skin.head()